# Run iQSM+ QSM reconstruction (Open Recon `qsm.py` module)

This mirrors [RunClientServerRecon.ipynb](RunClientServerRecon.ipynb), but wired specifically for
the `qsm.py` module: converts the example multi-echo GRE DICOMs (expected at
`data/DICOMs_openrecon`, gitignored -- populate it yourself) into MRD format, sends them to a
running server for QSM reconstruction via iQSM+ (https://github.com/sunhongfu/iQSM_Plus, cloned
locally into `iQSM_Plus/`, see [readme.md](readme.md)), and displays the result.

**Before running this notebook:** start the server using the **"Start QSM server"** launch
configuration (Run and Debug tab, top left) -- this runs `main.py -p 9020 -v -d qsm`, which
listens on port 9020 with `qsm` as the default config module.

## 1. Convert the example DICOMs to MRD format

In [ ]:
# Convert the example multi-echo GRE DICOMs (magnitude + phase, 160 slices x 5 echoes)
# into an MRD file. Expected at data/DICOMs_openrecon (gitignored, populate it yourself) --
# a repo-relative path, so this works the same in the devcontainer (workspace bind-mount
# covers the whole repo, including data/) and when running natively.
import os
import dicom2mrd
from types import SimpleNamespace

# Adjust the logging configuration otherwise INFO level isn't shown in notebook output
import logging
logging.getLogger().setLevel(logging.INFO)

dicomFolder = 'data/DICOMs_openrecon'
inputFilename = 'data/qsm_test.mrd'

dir, file = os.path.split(inputFilename)
if dir and not os.path.exists(dir):
    os.makedirs(dir)

if os.path.exists(inputFilename):
    overwrite = input(f'File {inputFilename} already exists!  Overwrite? y/[n]')
    if overwrite.lower() == 'y':
        os.remove(inputFilename)
    else:
        raise Exception('Aborting to not overwrite existing file')

args = SimpleNamespace(**dicom2mrd.defaults)
args.folder  = dicomFolder
args.outFile = inputFilename

dicom2mrd.main(args)

## 2. Run the QSM reconstruction

Make sure a QSM server is running before running this cell. Two options, both listen on port
9020 (this cell's `args.port` doesn't need to change either way):

- **"Start QSM server"** launch config (Run and Debug tab) -- runs `main.py` natively on macOS.
  Fast, but **cannot exercise bet2 brain extraction** -- bet2 is a linux/amd64 ELF binary baked
  into the Docker image and can't execute on macOS at all (`Exec format error`); `qsm.py`
  catches this and silently falls back to unmasked reconstruction (see `_run_bet2` in `qsm.py`).
- **"Start QSM server (Docker)"** task (Terminal > Run Task, or `Cmd+Shift+P` >
  "Tasks: Run Task") -- runs the actual `openrecon-qsm:prod` image, matching production. Slower
  to start (real container) but this is the only way to actually see bet2 run from this notebook.
  Requires the image to already be built (`docker build -f docker/qsm.dockerfile ...`, see that
  file's header comment) and Docker Desktop running.

Reconstruction takes several minutes (deep-learning inference over the full 3D multi-echo volume,
longer under Docker Desktop's `--platform linux/amd64` emulation on Apple Silicon) -- this cell
will block until it's done.

In [ ]:
# Send the converted data to the running QSM server
import client
from types import SimpleNamespace
import datetime
import logging
logging.basicConfig(format='%(asctime)s - %(message)s', level=logging.INFO)

outputFilename = 'data/qsm_recon.mrd'

args = SimpleNamespace(**client.defaults)
args.config    = 'qsm'
args.port      = 9020
args.out_group = str(datetime.datetime.now())
args.outfile   = outputFilename
args.filename  = inputFilename

client.main(args)


In [ ]:
import mrd2dicom
from types import SimpleNamespace

args = SimpleNamespace(
    filename   = outputFilename,   # reuses the variable from your reconstruction cell
    in_group   = '',               # '' -> auto-select most recent run
    out_folder = 'data/qsm_recon_dicom',
)
mrd2dicom.main(args)

## 3. Display the QSM map

In [ ]:
# Display the reconstructed QSM map (and the source magnitude for comparison)
import h5py
import ismrmrd
import numpy as np
import matplotlib.pyplot as plt

with h5py.File(outputFilename, 'r') as d:
    group = list(d.keys())[-1]  # most recent reconstruction run

with ismrmrd.Dataset(outputFilename, group, False) as dset:
    subgroups = dset.list()
    imgGroups = [g for g in subgroups if g.startswith('image_') or g.startswith('images_')]
    print('Group %s contains %d image series:' % (group, len(imgGroups)))
    for g in imgGroups:
        print(f'  {g} ({dset.number_of_images(g)} images)')

    # qsm.py writes the QSM map at image_series_index=100 ("image_100"), alongside the
    # original buffered magnitude/phase images it now also passes through unmodified (see
    # the "keep original DICOM series" change) as image_0/image_1/image_2/etc. Must select
    # by name, not by position: imgGroups[-1] used to work when image_100 was the only
    # series, but "image_2" now sorts *after* "image_100" as a string ('2' > '1'), so
    # picking "last" silently grabs a passthrough series instead of the QSM map.
    qsmGroup = 'image_100' if 'image_100' in imgGroups else imgGroups[-1]
    n = dset.number_of_images(qsmGroup)
    midImg = dset.read_image(qsmGroup, n // 2)
    meta = ismrmrd.Meta.deserialize(midImg.attribute_string)

# qsm.py stores the QSM map as quantized uint16 (see its RescaleSlope/RescaleIntercept
# comment) -- apply the Modality LUT transform (real = raw * slope + intercept) to recover
# real ppm values before plotting, the same conversion any DICOM viewer applies
# automatically. Without this, imshow's vmin/vmax (in ppm) clip every raw uint16 pixel
# to the same end of the colormap, rendering as a blank image.
slope     = float(meta['RescaleSlope'])
intercept = float(meta['RescaleIntercept'])
qsmPpm    = np.squeeze(midImg.data).astype(np.float64) * slope + intercept

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(qsmPpm, cmap='gray', vmin=-0.3, vmax=0.3)
ax.set_title(f'QSM (ppm), slice {n // 2 + 1}/{n}')
ax.axis('off')
plt.show()

print('MetaAttributes:')
for key, value in meta.items():
    print(f'  {key}: {value}')


## 4. Brain extraction test -- BET2 (FSL)

Tests optional brain-mask preprocessing for iQSM+ using FSL's `bet2`, vendored directly in the
repo at `vendor/bet2/{bin,lib}/` (~118 MB: `bet2` + its FSL-specific shared libraries, originally
resolved via `ldd` against the official `brainlife/fsl:6.0.7.22` image -- not a full FSL install,
which is 5+ GB). Vendored rather than re-extracted from that image at build/test time so there's
zero dependency on `brainlife/fsl` continuing to exist on Docker Hub (community-maintained, not
an official image -- see `vendor/bet2/README.md` for provenance/license/re-extraction details).

Tried [SynthStrip](https://github.com/nipreps/synthstrip) first (see git history for that
investigation): it failed to segment the cerebellum/temporal-pole region at all on this GRE
magnitude contrast (verified against both a synthetic and a geometrically-correct real-DICOM
affine, ruling out a test-harness artifact) -- a genuine model generalization limitation. Testing
BET2 as the classical (non-DL) alternative on the *same* problematic slices for direct comparison.

**Note:** `bet2` is a linux/amd64 ELF binary -- it cannot run directly from this notebook's Python
kernel on macOS. Cells below shell out to `docker run --platform linux/amd64` with `vendor/bet2/`
bind-mounted in, using a plain `python:3.12.0-slim` container (no FSL install) to prove the
vendored files actually work standalone without the full FSL environment.

This same vendored `bet2` is wired into `qsm.py` itself (see `_run_bet2`), toggled by the
"Brain Extraction (BET)" Open Recon UI parameter (default on) -- section 2 above will exercise it
for real if using the **"Start QSM server (Docker)"** task rather than the native launch config.

In [ ]:
# BET2 needs a plain 3D volume -- qsm.py's debug dump is 4D (echoes stacked on the last axis).
# Reuses the same single-echo extraction approach as the earlier SynthStrip test.
import os
import nibabel as nib
import numpy as np

debugMagPath = '/tmp/share/debug/mag.nii.gz'
if not os.path.exists(debugMagPath):
    raise FileNotFoundError(
        f'{debugMagPath} not found -- run the QSM reconstruction in section 2 first '
        '(qsm.py writes this debug file during process_qsm()).'
    )

betTestDir = 'data/bet2_test'
os.makedirs(betTestDir, exist_ok=True)

magImg = nib.load(debugMagPath)
magData = magImg.get_fdata()
print('Loaded', debugMagPath, '-- shape', magData.shape)

echo0 = magData[..., 0] if magData.ndim == 4 else magData
echo0Path = os.path.join(betTestDir, 'mag_echo0.nii.gz')
nib.save(nib.Nifti1Image(echo0.astype(np.float32), magImg.affine), echo0Path)
print('Saved single-echo volume for BET2:', echo0Path, echo0.shape)

In [ ]:
# Run bet2 via Docker (linux/amd64 binary, can't run natively on macOS). Bind-mounts the
# vendored vendor/bet2/ directory (bin + libs) and the test data folder into a plain
# python:3.12.0-slim container -- proves the vendored files work standalone, no FSL install.
import subprocess
import time

repoRoot = os.path.abspath('.')
betOutPrefix = os.path.join(betTestDir, 'mag_bet')

tic = time.perf_counter()
subprocess.run([
    'docker', 'run', '--rm', '--platform', 'linux/amd64',
    '-v', f'{repoRoot}/vendor/bet2:/opt/bet2:ro',
    '-v', f'{repoRoot}/{betTestDir}:/data',
    '-e', 'FSLOUTPUTTYPE=NIFTI_GZ',
    'python:3.12.0-slim',
    'sh', '-c',
    f'LD_LIBRARY_PATH=/opt/bet2/lib /opt/bet2/bin/bet2 /data/{os.path.basename(echo0Path)} '
    f'/data/{os.path.basename(betOutPrefix)} -m -f 0.5',
], check=True)
print('bet2 done in %.1f s' % (time.perf_counter() - tic))

maskPath = betOutPrefix + '_mask.nii.gz'
strippedPath = betOutPrefix + '.nii.gz'
print('Brain mask saved to:      ', maskPath)
print('Skull-stripped image to:  ', strippedPath)

In [ ]:
# Visualize: same slices used for the SynthStrip comparison (24/46/68 = problematic
# cerebellum/temporal-pole region, 91/113/136 = mid-brain, previously fine either way)
import matplotlib.pyplot as plt

mask = nib.load(maskPath).get_fdata()
print('Mask coverage: %.1f%% of voxels' % (100 * mask.sum() / mask.size))

sliceIdxs = [24, 46, 68, 91, 113, 136]

fig, axes = plt.subplots(2, 6, figsize=(24, 9))
for i, sl in enumerate(sliceIdxs):
    axes[0, i].imshow(echo0[:, :, sl].T, cmap='gray', origin='lower')
    axes[0, i].set_title(f'mag slice {sl}')
    axes[0, i].axis('off')

    axes[1, i].imshow(echo0[:, :, sl].T, cmap='gray', origin='lower')
    axes[1, i].contour(mask[:, :, sl].T, levels=[0.5], colors='red', linewidths=1.2)
    axes[1, i].set_title(f'+ BET2 mask slice {sl}')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()